# VQE & Quantum Chemistry

This notebook demonstrates:
1. Building a **molecular Hamiltonian** with `quantx_engine.chemistry`
2. Running **UCCSD-VQE** to find the ground-state energy
3. Running **noise-aware VQE** with the SPSA optimizer
4. Using **CVaR-QAOA** for MaxCut combinatorial optimization


In [ ]:
import numpy as np

## 1. Variational Quantum Eigensolver (UCCSD)

We construct a simple 2-qubit molecular Hamiltonian and minimise its expectation value
using the UCCSD ansatz.

In [ ]:
from quantx_engine.chemistry import PauliHamiltonian, PauliTerm
from quantx_engine.advanced_chemistry import UCCSDSolver, build_hamiltonian_matrix

# 2-qubit toy Hamiltonian: H = 0.5*ZI - 0.5*IZ + 0.25*XX + 0.25*YY
ham = PauliHamiltonian(n_qubits=2)
ham.add(PauliTerm(coeff= 0.50, ops={0: 'Z'}))
ham.add(PauliTerm(coeff=-0.50, ops={1: 'Z'}))
ham.add(PauliTerm(coeff= 0.25, ops={0: 'X', 1: 'X'}))
ham.add(PauliTerm(coeff= 0.25, ops={0: 'Y', 1: 'Y'}))

H_matrix = build_hamiltonian_matrix(ham)
exact_gs = np.linalg.eigvalsh(H_matrix)[0]
print(f'Exact ground-state energy: {exact_gs:.6f}')
print(f'Hamiltonian matrix shape:  {H_matrix.shape}')

In [ ]:
solver = UCCSDSolver(n_electrons=1, n_qubits=2)
result = solver.run(H_matrix, n_iter=100, step=0.1)

print(f'UCCSD-VQE energy: {result.energy:.6f}')
print(f'Converged:        {result.converged}')
print(f'Parameters used:  {result.n_params}')
# VQE should find the ground state
assert result.energy <= exact_gs + 0.01, f'VQE energy {result.energy} too far above exact {exact_gs}'

## 2. Noise-Aware VQE

The `noise_vqe` module wraps a hardware-efficient ansatz with the SPSA optimizer,
modelling shot noise and device-level gate errors.

In [ ]:
from quantx_engine.noise_vqe import NoiseAwareVQE

na_vqe = NoiseAwareVQE(
    hamiltonian=H_matrix,
    n_qubits=2,
    n_layers=2,
    max_iter=80,
    seed=42,
)
na_result = na_vqe.run()
print(f'Noise-aware VQE energy:     {na_result.energy:.6f}')
print(f'Iterations:                 {na_result.n_iterations}')
print(f'Noise description:          {na_result.noise_description}')
# Should be in the right ballpark
assert na_result.energy < 0.5, 'Noise-aware VQE energy should be negative'

## 3. CVaR-QAOA for MaxCut

QAOA optimises combinatorial problems. Here we solve a small MaxCut instance on a
triangle graph (3 nodes, 3 edges).  The optimal MaxCut value is 2.

In [ ]:
from quantx_engine.qaoa_optim import CVaRQAOA, MaxCutProblem

# Triangle graph: 3 nodes, 3 edges
problem = MaxCutProblem(n_nodes=3, edges=[(0, 1, 1.0), (1, 2, 1.0), (0, 2, 1.0)])
cost_diagonal = problem.cost_diagonal()
print(f'Cost diagonal: {cost_diagonal}')

# CVaR-QAOA: p=2 layers, CVaR with alpha=0.25 (top 25% samples)
qaoa = CVaRQAOA(
    cost_diagonal=cost_diagonal,
    n_qubits=3,
    n_layers=2,
    alpha=0.25,
)
result = qaoa.optimize(max_iter=100)

print(f'Best cut value found: {-result.optimal_value:.1f}  (optimal: 2)')
print(f'CVaR energy:          {result.expectation_value:.4f}')
# MaxCut of triangle = 2
assert -result.optimal_value >= 2, f'Expected MaxCut >= 2, got {-result.optimal_value}'
print('MaxCut assertion passed.')

## Summary

| Module | Demonstrated |
|--------|--------------|
| `advanced_chemistry.UCCSDSolver` | UCCSD-VQE ground-state energy optimisation |
| `noise_vqe.NoiseAwareVQE` | VQE with shot noise and SPSA optimizer |
| `qaoa_optim.CVaRQAOA` | CVaR-QAOA MaxCut on triangle graph |

Next: **[03 — Quantum Error Correction](03_quantum_error_correction.ipynb)**